# QLoRA Fine-tuning for Qwen2.5-Coder on Kaggle

**Instructions:**
1. Run this notebook on **Kaggle with GPU T4 x2 enabled** (Settings → Accelerator). Free ~30h/week available.
2. Upload this repo as a Kaggle dataset or `git clone` it.
3. Upload your `coding_sft.jsonl` dataset to Kaggle.

This notebook reuses the SFT training loop from the italian-llm repo to fine-tune Qwen2.5-Coder-1.5B-Instruct with QLoRA.

In [ ]:
!pip -q install -U transformers peft trl bitsandbytes accelerate datasets

In [ ]:
import os, sys
REPO = "/kaggle/working/italian-llm"   # adjust to your clone path
sys.path.insert(0, os.path.join(REPO, "src"))
DATA = "/kaggle/input/coding-sft/coding_sft.jsonl"  # adjust to your uploaded dataset

In [ ]:
from italian_llm.config import load_config
from italian_llm.training.sft import run_sft   # reuses the repo's SFT loop
cfg = load_config(os.path.join(REPO, "configs/train/sft_coder.yaml"))
cfg["data"]["train_path"] = DATA
cfg["train"]["output_dir"] = "/kaggle/working/sft_coder_lora"
output_dir = run_sft(cfg)

# Post-training: Merge & Export

After training completes:
1. Merge the LoRA adapter into the base model
2. Convert to GGUF using `scripts/quantize_gguf.py`
3. Download the `.gguf` file from Kaggle
4. On your local machine, run: `python scripts/export_ollama_coding.py --gguf <file>.gguf --name coder-local`

In [ ]:
# Optional: merge LoRA into base and save merged HF model for GGUF conversion
from italian_llm.training.common import load_model_and_tokenizer
from peft import PeftModel
from transformers import AutoTokenizer

# Load the base model and merge adapter
base, tokenizer = load_model_and_tokenizer({"model": {"name": "Qwen/Qwen2.5-Coder-1.5B-Instruct"}})
merged = PeftModel.from_pretrained(base, output_dir).merge_and_unload()
merged.save_pretrained("/kaggle/working/coder_merged")
tokenizer.save_pretrained("/kaggle/working/coder_merged")
print("Merged model at /kaggle/working/coder_merged — download and convert to GGUF.")